# 문제 17. 6개월 실현 LTV와 가치 집중도

## 0. 환경 설정

In [150]:
import pandas as pd
import numpy as np

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
pd.set_option("display.max_columns", 50)


## 1. 원본 데이터 적재

`orders.csv`, `order_items.csv`를 있는 그대로 읽어서 형태를 먼저 확인합니다.

In [151]:
orders = pd.read_csv("./data/orders.csv")
items = pd.read_csv("./data/order_items.csv")

print("orders shape:", orders.shape)
print("items shape:", items.shape)
orders.head()

orders shape: (200000, 5)
items shape: (500000, 6)


,order_id,customer_id,order_datetime,channel,status
0,25463,1077,2024-06-25 00:17:41,store,delivered
1,171088,2998,2024-05-28 19:35:20,web,delivered
2,27437,4066,2024-02-14 17:49:14,app,delivered
3,98425,3243,2024-05-08 16:37:36,store,delivered
4,22325,5331,2024-04-17 20:08:22,app,delivered


In [152]:
items.head()


,order_item_id,order_id,product_id,quantity,unit_price,discount
0,59256,114625,3,1,"20,300.00",0.05
1,230587,66337,80,2,"90,000.00",0.45
2,279813,163343,3,3,"20,300.00",0.23
3,88487,180332,79,1,"8,600.00",0.38
4,39240,174179,232,1,"42,600.00",0.31


## 2. 중복 제거

완전 중복 행을 제거하고, 몇 건이 사라졌는지 확인합니다.

In [153]:
n_orders_before, n_items_before = len(orders), len(items)

orders = orders.drop_duplicates()
items = items.drop_duplicates()

print(f"orders 중복 제거: {n_orders_before} -> {len(orders)} ({n_orders_before - len(orders)}건 손실)")
print(f"items  중복 제거: {n_items_before} -> {len(items)} ({n_items_before - len(items)}건 손실)")


orders 중복 제거: 200000 -> 200000 (0건 손실)
items  중복 제거: 500000 -> 499880 (120건 손실)


## 3. 날짜 파싱 (`errors="coerce"`)

타입 오염(빈 문자열 등)이 있는 행은 `NaT`로 바뀝니다. 몇 건이나 파싱에 실패했는지 확인합니다.

In [154]:
orders["order_datetime"] = pd.to_datetime(orders["order_datetime"], errors="coerce")

n_nat = orders["order_datetime"].isna().sum()
print(f"날짜 파싱 실패(NaT) 건수: {n_nat}")
orders[orders["order_datetime"].isna()].head()


날짜 파싱 실패(NaT) 건수: 1933


,order_id,customer_id,order_datetime,channel,status
40,44572,1763,NaT,web,paid
161,163116,1525,NaT,web,paid
195,196024,3906,NaT,web,returned
262,192106,1802,NaT,web,delivered
263,174676,2097,NaT,web,shipped


## 4. 기간 필터 (2024년 상반기: 1/1 ~ 6/30)

`common_order_items.py`와 동일하게 `2024-01-01 <= order_datetime < 2024-07-01`만 남깁니다.
NaT는 비교에서 자동으로 False가 되어 이 단계에서 함께 걸러집니다.

In [155]:
period_mask = (orders["order_datetime"] >= "2024-01-01") & (orders["order_datetime"] < "2024-07-01")
orders_in_period = orders[period_mask].copy()

print(f"기간 필터 전: {len(orders):,}건")
print(f"기간 필터 후: {len(orders_in_period):,}건 (제외 {len(orders) - len(orders_in_period):,}건, NaT 포함)")
orders_in_period["order_datetime"].agg(["min", "max"])


기간 필터 전: 200,000건
기간 필터 후: 198,027건 (제외 1,973건, NaT 포함)


min   2024-01-01 00:02:25
max   2024-06-30 23:57:48
Name: order_datetime, dtype: datetime64[us]

## 5. orders × order_items 결합 (inner join)

`order_id` 기준으로 결합합니다. 결합 전후 건수를 비교해 유실이 없는지 확인합니다.

In [156]:
# orders_in_period (2024-01-01 ~ 2024-06-30) 기준으로 items와 결합
order_items_period = orders_in_period.merge(items, on="order_id", how="inner")

print(f"결합 전 order_items_period(전체): {len(items):,}건")
print(f"결합 후 order_items_period(기간 내 주문만): {len(order_items_period):,}건")
order_items_period.head()



결합 전 order_items_period(전체): 499,880건
결합 후 order_items_period(기간 내 주문만): 494,994건


,order_id,customer_id,order_datetime,channel,status,order_item_id,product_id,quantity,unit_price,discount
0,25463,1077,2024-06-25 00:17:41,store,delivered,265800,104,5,"73,300.00",0.40
1,25463,1077,2024-06-25 00:17:41,store,delivered,16468,319,4,"68,800.00",0.21
2,25463,1077,2024-06-25 00:17:41,store,delivered,477391,9,4,"19,400.00",0.34
3,171088,2998,2024-05-28 19:35:20,web,delivered,55193,279,4,"28,000.00",0.02
4,171088,2998,2024-05-28 19:35:20,web,delivered,200065,3,2,"20,300.00",0.32


## 6. `unit_price` 결측 제외 + `line_amount` 파생

`line_amount = quantity × unit_price × (1 - discount)`

In [ ]:
n_before = len(order_items_period)

order_items = order_items_period.dropna(subset=["unit_price"])
# print(f"unit_price 결측 제외: {n_before:,} -> {len(order_items_period):,}건 ({n_before - len(order_items_period):,}건 손실)")

# order_items["line_amount"] = (
#     order_items_period["quantity"] * order_items_period["unit_price"] * (1 - order_items_period["discount"])
# )
print('order_items1 -> \n',order_items.head())
order_items[["order_id", "quantity", "unit_price", "discount", "line_amount"]].head()
print('order_items2 -> \n',order_items)




order_items1 -> 
    order_id  customer_id      order_datetime channel     status  \
0     25463         1077 2024-06-25 00:17:41   store  delivered   
1     25463         1077 2024-06-25 00:17:41   store  delivered   
2     25463         1077 2024-06-25 00:17:41   store  delivered   
3    171088         2998 2024-05-28 19:35:20     web  delivered   
4    171088         2998 2024-05-28 19:35:20     web  delivered   

   order_item_id  product_id  quantity  unit_price  discount  is_net  \
0         265800         104         5   73,300.00      0.40    True   
1          16468         319         4   68,800.00      0.21    True   
2         477391           9         4   19,400.00      0.34    True   
3          55193         279         4   28,000.00      0.02    True   
4         200065           3         2   20,300.00      0.32    True   

   line_amount  
0   219,900.00  
1   217,408.00  
2    51,216.00  
3   109,760.00  
4    27,608.00  
order_items2 -> 
         order_id  customer

## 7. 정상건(`is_net`) 플래그

`canceled`, `returned` 상태를 제외한 나머지가 순매출(net) 라인입니다.

In [172]:
# print(order_items.head())

order_items_period["is_net"] = ~order_items_period["status"].isin(["canceled", "returned"])

print(order_items_period["status"].value_counts())
print(order_items_period)
print(order_items_period["is_net"].value_counts())




status
delivered    271543
paid          74535
shipped       74287
canceled      50036
returned      24593
Name: count, dtype: int64
        order_id  customer_id      order_datetime channel     status  \
0          25463         1077 2024-06-25 00:17:41   store  delivered   
1          25463         1077 2024-06-25 00:17:41   store  delivered   
2          25463         1077 2024-06-25 00:17:41   store  delivered   
3         171088         2998 2024-05-28 19:35:20     web  delivered   
4         171088         2998 2024-05-28 19:35:20     web  delivered   
...          ...          ...                 ...     ...        ...   
494989     62264         2403 2024-03-21 00:22:40     web  delivered   
494990     62264         2403 2024-03-21 00:22:40     web  delivered   
494991     62264         2403 2024-03-21 00:22:40     web  delivered   
494992      1963         3439 2024-01-01 00:58:02     app   canceled   
494993      1963         3439 2024-01-01 00:58:02     app   canceled   

  

## 8. 순매출 라인만 추출 (`net_order_items()`에 해당)

In [171]:
net_items = order_items_period[order_items_period["is_net"]].copy()
print(f"전체 라인: {len(order_items_period):,}건 / 순매출 라인: {len(net_items):,}건")
net_items.head()


전체 라인: 494,994건 / 순매출 라인: 420,365건


,order_id,customer_id,order_datetime,channel,status,order_item_id,product_id,quantity,unit_price,discount,is_net
0,25463,1077,2024-06-25 00:17:41,store,delivered,265800,104,5,"73,300.00",0.40,True
1,25463,1077,2024-06-25 00:17:41,store,delivered,16468,319,4,"68,800.00",0.21,True
2,25463,1077,2024-06-25 00:17:41,store,delivered,477391,9,4,"19,400.00",0.34,True
3,171088,2998,2024-05-28 19:35:20,web,delivered,55193,279,4,"28,000.00",0.02,True
4,171088,2998,2024-05-28 19:35:20,web,delivered,200065,3,2,"20,300.00",0.32,True


---
## 9. 상품 원가(cost) 결합

판매 단가는 오염된 `products.price`가 아니라 거래 시점 값인 `order_items.unit_price`를 그대로 씁니다.
여기서는 원가(`cost`)만 상품 마스터에서 가져옵니다.

In [160]:
products = pd.read_csv("data/products.csv").drop_duplicates()
products.head()


,product_id,product_name,category,price,cost
0,119,플러스 홍차,식품,0,"3,700.00"
1,95,프리미엄 노트북,전자,68600.0,"28,900.00"
2,112,프리미엄 원두커피,식품,10500.0,"4,700.00"
3,164,베이직 마우스,전자,52100.0,"32,600.00"
4,274,베이직 홍차,식품,16000.0,"10,500.00"


In [170]:
items_pro = net_items.merge(products[["product_id", "cost"]], on="product_id", how="left")

# 결측치 확인
n_cost_null = items_pro["cost"].isna().sum()
print(f"원가(cost) 결측 라인: {n_cost_null:,}건")
items_pro[["product_id", "unit_price", "cost"]].head()
print(items_pro)


원가(cost) 결측 라인: 0건
        order_id  customer_id      order_datetime channel     status  \
0          25463         1077 2024-06-25 00:17:41   store  delivered   
1          25463         1077 2024-06-25 00:17:41   store  delivered   
2          25463         1077 2024-06-25 00:17:41   store  delivered   
3         171088         2998 2024-05-28 19:35:20     web  delivered   
4         171088         2998 2024-05-28 19:35:20     web  delivered   
...          ...          ...                 ...     ...        ...   
420360     88502         3265 2024-01-20 06:44:00     app  delivered   
420361     88502         3265 2024-01-20 06:44:00     app  delivered   
420362     62264         2403 2024-03-21 00:22:40     web  delivered   
420363     62264         2403 2024-03-21 00:22:40     web  delivered   
420364     62264         2403 2024-03-21 00:22:40     web  delivered   

        order_item_id  product_id  quantity  unit_price  discount  is_net  \
0              265800         104      

## 10. 라인별 마진(margin) 계산

`margin = quantity × (unit_price × (1 - discount) - cost)`

In [174]:
items_pro["margin"] = items_pro["quantity"] * (
    items_pro["unit_price"] * (1 - items_pro["discount"]) - items_pro["cost"]
)
items_pro["line_amount"] = (
    items_pro["quantity"] * items_pro["unit_price"] * (1 - items_pro["discount"])
)

items_pro.head()
items_pro[["quantity", "unit_price", "discount", "cost", "line_amount", "margin"]].head()


,quantity,unit_price,discount,cost,line_amount,margin
0,5,"73,300.00",0.40,"49,200.00","219,900.00","-26,100.00"
1,4,"68,800.00",0.21,"40,900.00","217,408.00","53,808.00"
2,4,"19,400.00",0.34,"11,000.00","51,216.00","7,216.00"
3,4,"28,000.00",0.02,"15,400.00","109,760.00","48,160.00"
4,2,"20,300.00",0.32,"10,500.00","27,608.00","6,608.00"


## 11. 고객별 매출LTV / 마진LTV 집계

In [199]:
cust_ltv = items_pro.groupby("customer_id").agg(
    매출LTV=("line_amount", "sum"),
    마진LTV=("margin", "sum"),
)

print(f"고객 수: {len(cust_ltv):,}명")
cust_ltv.head()


고객 수: 5,720명


,매출LTV,마진LTV
customer_id,,
1000,"2,530,766.00","790,466.00"
1001,"5,849,515.00","785,615.00"
1002,"2,432,546.00","549,846.00"
1003,"18,760,198.00","3,777,498.00"
1004,"7,727,622.00","1,169,122.00"


## 12. 분포 요약 (평균 / 중앙값 / P90 / P99)

In [189]:
def dist_stats(s):
    return pd.Series({
        "평균": s.mean(),
        "중앙값": s.median(),
        "P90": s.quantile(0.9),
        "P99": s.quantile(0.99),
    })

dist_summary = pd.DataFrame({
    "매출LTV": dist_stats(cust_ltv["매출LTV"]),
    "마진LTV": dist_stats(cust_ltv["마진LTV"]),
})
dist_summary


,매출LTV,마진LTV
평균,"8,148,395.38","1,866,809.74"
중앙값,"3,590,926.00","819,871.50"
P90,"12,929,847.10","3,020,856.80"
P99,"78,586,392.62","18,577,423.72"


## 13. 상위 10% 고객 집중도

In [203]:
top10_share = (
    cust_ltv["매출LTV"].sort_values(ascending=False)
    .head(int(len(cust_ltv) * 0.1)).sum()
    / cust_ltv["매출LTV"].sum()
)
# 고객별 매출LTV / 마진LTV
print(len(cust_ltv))
print(cust_ltv["매출LTV"].sort_values(ascending=False).head(int(len(cust_ltv) * 0.1)).sum())
print(int(len(cust_ltv) * 0.1))
print(cust_ltv["매출LTV"].sum())
# print(f"상위 10% 고객이 전체 매출LTV의 {top10_share:.1%}를 차지")


5720
27143962880.0
572
46608821601.0


## 14. 평균-중앙값 간극

In [192]:
# 	          매출LTV	     마진LTV
# 평균	 8,148,395.38	1,866,809.74
# 중앙값 3,590,926.00	  819,871.50
# P90	12,929,847.10	3,020,856.80
# P99	78,586,392.62	18,577,423.72

gap = dist_summary.loc["평균", "매출LTV"] / dist_summary.loc["중앙값", "매출LTV"] - 1
print(f"평균이 중앙값보다 {gap:+.1%} 높음")


평균이 중앙값보다 +126.9% 높음


## 15. 고객 모수 확인 (마스터 대비)

In [201]:
customers = pd.read_csv("./data/customers.csv")

ltv_ids = set(cust_ltv.index)
master_ids = set(customers["customer_id"])

only_in_ltv = ltv_ids - master_ids      # LTV에는 있는데 마스터엔 없는 고객
only_in_master = master_ids - ltv_ids   # 마스터엔 있는데 LTV엔 없는 고객(구매 이력 없음)

n_ltv_customers = len(ltv_ids)          # LTV에 있는 고객 명수
n_master_customers = len(master_ids)    # MASTER에 있는 고객 명수

print(f"LTV 계산에 등장한 고객 수: {n_ltv_customers:,}명")
print(f"고객 마스터 등록 고객 수: {n_master_customers:,}명")
print(f"LTV에는 있으나 고객 마스터에는 없는 고객: {len(only_in_ltv):,}명")
print(f"고객 마스터에는 있으나 LTV(구매 이력)에는 없는 고객: {len(only_in_master):,}명")

LTV 계산에 등장한 고객 수: 5,720명
고객 마스터 등록 고객 수: 4,950명
LTV에는 있으나 고객 마스터에는 없는 고객: 770명
고객 마스터에는 있으나 LTV(구매 이력)에는 없는 고객: 0명


## 16. 결론 정리

In [ ]:
print(
    f"[시사점] 평균이 중앙값보다 {gap:+.1%} 높고 상위 10% 고객이 매출LTV의 "
    f"{top10_share:.1%}를 차지한다는 것은 소수 고액 고객이 평균을 끌어올렸다는 뜻이다.\n"
    "따라서 '평균 LTV x N배수'식 획득비 상한은 상위 소수 고객에 맞춰진 과대 기준일 위험이 크며,\n"
    "중앙값이나 코호트별 곡선(문제 18) 같은 왜곡에 강한 지표로 다시 세워야 한다."
)

[시사점] 평균이 중앙값보다 +126.9% 높고 상위 10% 고객이 매출LTV의 58.2%를 차지한다는 것은 소수 고액 고객이 평균을 끌어올렸다는 뜻이다.
따라서 '평균 LTV x N배수'식 획득비 상한은 상위 소수 고객에 맞춰진 과대 기준일 위험이 크며,
중앙값이나 코호트별 곡선(문제 18) 같은 왜곡에 강한 지표로 다시 세워야 한다.
